In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from pandas.api.types import is_datetime64_any_dtype, is_numeric_dtype

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# -------------------------------------------------------------------------
# Bulk CSV Loader with Smart Encoding Handling
#
# What this script does:
# 1. Sets input (RAW_DIR) and output (OUT_DIR) directories.
# 2. Recursively finds and loads all CSV files from RAW_DIR.
# 3. Handles tricky encodings (UTF-8 with BOM, fallback to Latin-1).
# 4. Keeps loading even if some files fail, and reports errors.
# 5. Stores loaded CSVs in a dictionary (tables) keyed by relative file name.
# 6. Prints a summary: number of CSVs loaded and their names.
# -------------------------------------------------------------------------

# Define input and output directories
current_user = os.getlogin()
if current_user == "bouba.ismalia":
    RAW_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - data")
else:
    RAW_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - Analytics\Student drop-out\data")

OUT_DIR = Path(rf"{RAW_DIR}\processed")

# --- Helpers ---
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV with sensible defaults, trying UTF-8 first and Latin-1 as fallback."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)


def load_all_csvs(base: Path, **kwargs) -> dict[str, pd.DataFrame]:
    """
    Load ALL CSVs under base (recursively) into a dict keyed by relative path (without .csv).
    Prints how many tables were loaded and how many errors occurred.
    """
    tables, errors = {}, []
    for p in sorted(base.rglob("*.csv")):
        key = str(p.relative_to(base).with_suffix(""))  # e.g. "sub/filename"
        try:
            tables[key] = read_csv_smart(p, **kwargs)
        except Exception as e:
            errors.append((str(p), str(e)))
    print(f"Loaded {len(tables)} tables; errors: {len(errors)}")
    if errors:
        for name, err in errors[:10]:
            print(" -", name, "->", err)
    return tables


# --- Load all CSV files ---
tables = load_all_csvs(
    RAW_DIR,
    sep=None,        # auto-detect delimiter
    engine="python", # required for sep=None sniffing
)

# Print summary in the requested format
print(f"\nNumber of csv files found: {len(tables)}")
print("List of csv files:")
for i, name in enumerate(tables.keys(), start=1):
    print(f"{i}. {name}")


In [ ]:
# show summary of data files
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)   # don't truncate lists

summary = (
    pd.DataFrame(
        [{
            "key": k,
            "rows": len(df),
            "cols": df.shape[1],
            "columns": list(map(str, df.columns))  # full list, no manual truncation
        } for k, df in tables.items()]
    ).sort_values("key")
)

summary


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Create globally accessible pandas DataFrames with clean, consistent
# names from a `tables` source dict. For each (clean_name -> source_key) pair,
# this script fetches the DataFrame from `tables`, assigns it into `globals()`
# under `clean_name`, and logs what was created. If a source key is missing,
# it logs a warning.
# -----------------------------------------------------------------------------

# Step 1: Map clean DataFrame names -> source keys in `tables`
custom_names = {
    "sdo1_distance": "sdo1-helperdata-Euclidean_distance_postcode_to_HU",
    "sdo1_characteristics": "sdo1-student_characteristics",
    "sdo1_previous": "sdo1-student_previous_education",
    "sdo2_skc": "sdo2-skc",
    "sdo2_orientation": "sdo2-student_orientation",
    "sdo4_dsa": "sdo4-DSA_degree_collegeyear_2018-2023",
    "sdo5_degree": "sdo5-student_degree_drop-out_postalcode",
    "sdo6_results": "sdo6-course_results",
    "sdo7_questionaire": "sdo78_sample-data",
}

# Step 2: Create DataFrames with clean names (keep everything)
for clean_name, source_key in custom_names.items():
    df = tables.get(source_key)
    if df is None:
        print(f"[warning] '{source_key}' not found in tables.")
        continue

    globals()[clean_name] = df
    print(f"Created DataFrame: {clean_name} from '{source_key}' with {len(df)} rows")


------------------------- For each dataframe, drop rows that are entirely empty across all columns ----------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: For each loaded DataFrame, convert empty/whitespace-only strings to NaN
# (object/string columns only) and DROP rows that are entirely empty (all-NaN).
# Prints a before/after summary per dataset. Safe to run multiple times.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
]

def _drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    # Treat pure-whitespace as empty only for textual columns
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols):
        df = df.copy()
        df[obj_cols] = df[obj_cols].replace(r"^\s*$", np.nan, regex=True)

    empty_mask = df.isna().all(axis=1)
    n_empty = int(empty_mask.sum())
    return df.loc[~empty_mask].copy() if n_empty else df, n_empty

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    before = len(df)
    cleaned, dropped = _drop_fully_empty_rows(df)
    globals()[name] = cleaned
    after = len(cleaned)

    print(f"{name}: dropped {dropped} fully-empty rows (rows {before} → {after})")

# Assert no fully empty rows remain in any dataset
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        assert not df.isna().all(axis=1).any(), f"{name} still has fully empty rows."


Anticipation: 

Fully empty rows usually come from source files (especially Excel) where the sheet’s UsedRange includes formatted-but-empty rows or trailing delimiters, 
so the importer reads them as blank records. They can also be created downstream by outer joins/concats with mismatched keys or 
by converting whitespace/empty strings to NaN across all columns, leaving rows with nothing but missing values.

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Drop any "unnamed" columns (e.g., "Unnamed: 0", empty-string headers)
# from all loaded DataFrames. Safe to run multiple times; prints what was removed.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
]

def _unnamed_cols(df: pd.DataFrame):
    cols = []
    for c in df.columns:
        s = str(c)
        if s.strip() == "" or s.lower().startswith("unnamed"):
            cols.append(c)
    return cols

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    to_drop = _unnamed_cols(df)
    if to_drop:
        globals()[name] = df.drop(columns=to_drop)
        print(f"{name}: dropped {len(to_drop)} unnamed columns -> {to_drop}")
    else:
        print(f"{name}: no unnamed columns found.")


# Data Overview: schema and Sanity check
Verifying that each dataset loaded matches the structure (“schema”) as expected

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None

def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        # If invalid, capture a small sample of duplicates for inspection
        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        # No SINH_ID present (e.g., sdo1_distance, sdo4_dsa) — search for composite
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports ---
pk_report = (
    pd.DataFrame(pk_rows)
    .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
    .reset_index(drop=True)
)

# Optional: a quick, filtered view of what you likely want to confirm
expected_pk_view = pk_report.loc[pk_report["has_SINH_ID"], ["dataset", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]]

# Emit logs so you can see what got renamed and any dup samples available
if rename_log:
    print("Renamed headers to normalize SINH_ID:")
    for ds, old, new in rename_log:
        print(f"  - {ds}: '{old}' -> '{new}'")

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)

# If you want to quickly inspect duplicate SINH_IDs:
for ds_name, dup_df in dup_samples.items():
    print(f"\nTop duplicated SINH_IDs in {ds_name}:")
    display(dup_df)

# Example: enforce index for frames with valid single-column PK
for ds_name, df in df_registry.items():
    if "SINH_ID" in df.columns:
        if df["SINH_ID"].isna().sum() == 0 and df.duplicated(subset=["SINH_ID"]).sum() == 0:
            try:
                df.set_index("SINH_ID", inplace=True, drop=False)  # keep column + index
                print(f"Index set to SINH_ID for {ds_name}")
            except Exception as e:
                print(f"[warning] Could not set index for {ds_name}: {e}")


# Based on the above results, handle tables with invalid primary key:
sdo1_previous,
sdo2_orientation

-------------------------------------------------- sdo1_previous ----------------------------------------------------

In [ ]:
sdo1_previous

In [ ]:
sdo1_previous.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Make SINH_ID the primary key in sdo1_previous by keeping only the row
# with the LATEST 'Final Exam Date' per SINH_ID. Rows with missing SINH_ID are dropped.
# Dates are parsed day-first (e.g., 28-06-2017). NaT dates are never selected when a
# real date exists for that SINH_ID.
# -----------------------------------------------------------------------------

df = sdo1_previous.copy()

# Parse dates (European format) and standardize SINH_ID
df["Final Exam Date"] = pd.to_datetime(df["Final Exam Date"], dayfirst=True, errors="coerce")
df["SINH_ID"] = pd.to_numeric(df["SINH_ID"], errors="coerce").round().astype("Int64")

# Drop rows without SINH_ID (cannot be a PK)
df = df.dropna(subset=["SINH_ID"]).copy()

# Sort so real dates come after NaT; keeping 'last' selects the latest real date
df = df.sort_values(["SINH_ID", "Final Exam Date"], ascending=[True, True], na_position="first")

# Keep only the latest row per SINH_ID
sdo1_previous = df.drop_duplicates(subset=["SINH_ID"], keep="last").copy()

# Sanity checks
assert sdo1_previous["SINH_ID"].isna().sum() == 0, "SINH_ID still has NaNs after filtering."
assert sdo1_previous.duplicated(subset=["SINH_ID"]).sum() == 0, "SINH_ID not unique after deduping."

print("sdo1_previous now unique on SINH_ID with latest Final Exam Date per student.")
print("Rows:", len(sdo1_previous))


In [ ]:
sdo1_previous

------------------------------------------ sdo2_orientation ------------------------------------------

In [ ]:
sdo2_orientation

In [ ]:
# For the sdo2_orientation data frame, check how many NaN are there in the SINH_ID column

nan_count = int(sdo2_orientation["SINH_ID"].isna().sum())
print(nan_count)

In [ ]:
# Check the count of rows with a NaN in at least two columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 2).sum())
print(count_at_least_two_nan)


In [ ]:

# Check the count of rows with a NaN in at least three columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 3).sum())
print(count_at_least_two_nan)


In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
print(sdo2_orientation["Event_Types_Attended"].value_counts(dropna=False))

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Aggregate sdo2_orientation by SINH_ID, summing attendance metrics.
# Result has exactly three columns: SINH_ID, Number_of_Events_Attended, Number_of_Event_Types.
# -----------------------------------------------------------------------------

cols = ["SINH_ID", "Number_of_Events_Attended", "Number_of_Event_Types"]
df = sdo2_orientation[cols].copy()

# Ensure numeric for safe summation
df["Number_of_Events_Attended"] = pd.to_numeric(df["Number_of_Events_Attended"], errors="coerce")
df["Number_of_Event_Types"] = pd.to_numeric(df["Number_of_Event_Types"], errors="coerce")

sdo2_orientation = (
    df.groupby("SINH_ID", as_index=False)
      .agg(
          Number_of_Events_Attended=("Number_of_Events_Attended", "sum"),
          Number_of_Event_Types=("Number_of_Event_Types", "sum"),
      )
)

# Optional: preview
sdo2_orientation


In [ ]:
print(sdo2_orientation["Number_of_Events_Attended"].value_counts(dropna=False))

In [ ]:
print(sdo2_orientation["Number_of_Event_Types"].value_counts(dropna=False))

In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

-------------------------------------- sdo1_distance -----------------------------------------------

In [ ]:
# Next steps'
# Check how to combine sdo1_distance and sdo4_dsa with the other data frames.
# Is STUDENTNUMMER in sdo2_orientation same as id in sdo1_distance? 
Bouba

# Check for primary key recommendations:
Keep a tighter approach in mind,  eg. checking for the float values duplicates eg 12345.0 vs 12345, make it integers.

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Check for primary key recommendations: 
# Enforce a consistent dtype for SINH_ID, then (per DataFrame) find the
# minimal primary key. Prefer single-column SINH_ID where valid; otherwise search
# for the *smallest* composite key that includes SINH_ID (pairs, then triples).
# For datasets without SINH_ID, reuse the discovered composite keys you found.
# Emits a concise recommendation table and (optionally) sets indexes safely.
# -----------------------------------------------------------------------------

# --- 0) Registry (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {n: globals()[n] for n in clean_names if isinstance(globals().get(n), pd.DataFrame)}

# --- 1) Normalize/standardize SINH_ID dtype wherever present ---
def _canonicalize(s: str) -> str:
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def normalize_sinh_id_inplace(df: pd.DataFrame) -> bool:
    """
    If a SINH_ID-like column exists, rename to 'SINH_ID' and coerce to pandas
    nullable integer ('Int64'), preserving NA. Returns True if column exists.
    """
    col = find_sinh_id_column(df.columns)
    if not col:
        return False
    if col != "SINH_ID":
        df.rename(columns={col: "SINH_ID"}, inplace=True)

    # Coerce float/str -> Int64 (nullable). This keeps NaN as <NA>.
    # If there are non-integer strings, they become <NA>.
    if not is_integer_dtype(df["SINH_ID"]):
        df["SINH_ID"] = pd.to_numeric(df["SINH_ID"], errors="coerce")
        # If values are like 12345.0, astype('Int64') will floor only if all are integers.
        # We round first to be tolerant, then coerce again.
        df["SINH_ID"] = pd.to_numeric(df["SINH_ID"].round(0), errors="coerce").astype("Int64")
    return True

for name, df in df_registry.items():
    _ = normalize_sinh_id_inplace(df)

# --- 2) Helper to check uniqueness for a candidate key ---
def key_diagnostics(df: pd.DataFrame, cols: list[str]) -> dict:
    sub = df[cols]
    null_rows = int(sub.isna().any(axis=1).sum())
    dup_rows  = int(df.duplicated(subset=cols).sum())
    return {
        "pk_columns": tuple(cols),
        "null_rows": null_rows,
        "dup_rows": dup_rows,
        "is_valid_pk": (null_rows == 0 and dup_rows == 0),
    }

# --- 3) For each dataset, determine the minimal PK ---
recommendations = []

for name, df in df_registry.items():
    # Default: unknown
    rec = {"dataset": name, "pk_columns": None, "pk_type": None, "null_rows": None, "dup_rows": None, "note": ""}

    # A) Special cases without SINH_ID (use previously validated pairs)
    if "SINH_ID" not in df.columns:
        if name == "sdo1_distance":
            diag = key_diagnostics(df, ["id", "postcode"])
            rec.update({"pk_columns": diag["pk_columns"], "pk_type": "pair", "null_rows": diag["null_rows"], "dup_rows": diag["dup_rows"]})
            rec["note"] = "No SINH_ID; using (id, postcode)."
        elif name == "sdo4_dsa":
            diag = key_diagnostics(df, ["CROHO naam", "Collegejaar"])
            rec.update({"pk_columns": diag["pk_columns"], "pk_type": "pair", "null_rows": diag["null_rows"], "dup_rows": diag["dup_rows"]})
            rec["note"] = "No SINH_ID; using (CROHO naam, Collegejaar)."
        else:
            rec["note"] = "No SINH_ID present; please define table grain."
        recommendations.append(rec)
        continue

    # B) Try single-column SINH_ID
    diag_single = key_diagnostics(df, ["SINH_ID"])
    if diag_single["is_valid_pk"]:
        rec.update({"pk_columns": diag_single["pk_columns"], "pk_type": "single", "null_rows": diag_single["null_rows"], "dup_rows": diag_single["dup_rows"]})
        recommendations.append(rec)
        continue

    # C) Otherwise, search smallest composite key including SINH_ID
    cols = [c for c in df.columns if c != "SINH_ID"]
    found = None

    # pairs (SINH_ID + x)
    for c in cols:
        diag = key_diagnostics(df, ["SINH_ID", c])
        if diag["is_valid_pk"]:
            found = ("pair", diag)
            break

    # triples (SINH_ID + x + y), if needed
    if not found and len(cols) >= 2:
        for c1, c2 in combinations(cols, 2):
            diag = key_diagnostics(df, ["SINH_ID", c1, c2])
            if diag["is_valid_pk"]:
                found = ("triple", diag)
                break

    if found:
        pk_type, diag = found
        rec.update({"pk_columns": diag["pk_columns"], "pk_type": pk_type, "null_rows": diag["null_rows"], "dup_rows": diag["dup_rows"]})
        rec["note"] = "SINH_ID not unique alone; minimal composite found."
    else:
        rec.update({"pk_columns": ("SINH_ID",), "pk_type": "single", "null_rows": diag_single["null_rows"], "dup_rows": diag_single["dup_rows"]})
        rec["note"] = "No unique composite up to size 3; table grain likely multi-row per SINH_ID."

    recommendations.append(rec)

pk_recommendations = (
    pd.DataFrame(recommendations)
      .sort_values("dataset")
      .reset_index(drop=True)
)

print("Primary key recommendations (per dataset):")
display(pk_recommendations[["dataset","pk_type","pk_columns","null_rows","dup_rows","note"]])

# --- 4) (Optional) Set indexes where valid and safe (won't error if not unique) ---
for _, row in pk_recommendations.iterrows():
    ds = row["dataset"]; cols = list(row["pk_columns"] or [])
    if not cols:
        continue
    df = df_registry[ds]
    diag = key_diagnostics(df, cols)
    if diag["is_valid_pk"]:
        try:
            df.set_index(cols, inplace=True, drop=False)  # keep key columns as columns
            print(f"Index set for {ds}: {cols}")
        except Exception as e:
            print(f"[warning] Could not set index for {ds} -> {cols}: {e}")


------------------- Check for duplicate SINH_ID, other columns and in rows---------------------------------------

In [ ]:
# Checks each dataset for duplicate primary keys, handling key variants ('SINH_ID' / 'sinh_id' / 'sinh_d').
# Prints a per-dataset summary (rows, unique IDs, duplicate rows/IDs) and previews the most frequent duplicates.
# If duplicates exist, automatically deduplicates on the key using KEEP_POLICY (default: keep='last') and verifies the result.
# Stores all removed duplicate rows in `removed_dups` for auditability; keeps cleaned DataFrames in `datasets`.
# Reassigns the cleaned DataFrames back to the original variables (sdo1_characteristics, sdo2_skc, sdo5_degree, sdo6_course_results).
# Tip: change KEEP_POLICY to 'first' if that better matches your convention; add/remove datasets in the `datasets` dict as needed.


def find_key_col(df: pd.DataFrame) -> str:
    """Find the key column (handles SINH_ID / sinh_id / sinh_d)."""
    cmap = {c.lower(): c for c in df.columns}
    for cand in ("sinh_id", "sinh_d"):
        if cand in cmap:
            return cmap[cand]
    raise KeyError("No SINH_ID/sinh_id/sinh_d column found.")

def dup_summary(df: pd.DataFrame, key: str) -> dict:
    return {
        "rows": len(df),
        "unique_ids": df[key].nunique(dropna=True),
        "dup_rows": int(df.duplicated(subset=[key], keep=False).sum()),
        "dup_ids": int((df[key].value_counts(dropna=True) > 1).sum()),
    }

def dedup_df(df: pd.DataFrame, key: str, keep: str = "last"):
    """Return (clean_df, dropped_rows_df) by deduplicating on key."""
    mask_dups_to_drop = df.duplicated(subset=[key], keep=keep)
    dropped = df.loc[mask_dups_to_drop].copy()
    clean = df.drop_duplicates(subset=[key], keep=keep)
    return clean, dropped

datasets = {
    "sdo1_characteristics": sdo1_characteristics,
    "sdo2_skc": sdo2_skc,
    #// you didn't include sdo4?
    "sdo5_degree": sdo5_degree,
    "sdo6_course_results": sdo6_course_results,
}

removed_dups = {}   # keep dropped rows for audit
KEEP_POLICY = "last"  # change to "first" if preferred

for name, df in datasets.items():
    key = find_key_col(df)
    s = dup_summary(df, key)
    print(f"\n{name}: key='{key}' | rows={s['rows']:,} | unique_ids={s['unique_ids']:,} | "
          f"duplicate_rows={s['dup_rows']:,} | duplicate_ids={s['dup_ids']:,}")

    # show a quick peek of top duplicate IDs (if any)
    if s["dup_ids"] > 0:
        top_dups = (
            df.groupby(key, dropna=False).size()
              .reset_index(name="count")
              .query("count > 1")
              .sort_values("count", ascending=False)
              .head(10)
        )
        print("Top duplicate IDs:")
        print(top_dups.to_string(index=False))

    # ---- IF condition to ensure duplicates are handled ----
    if s["dup_ids"] > 0:
        print(f"-> Deduplicating {name} on '{key}' (keep='{KEEP_POLICY}')")
        clean, dropped = dedup_df(df, key, keep=KEEP_POLICY)
        datasets[name] = clean
        removed_dups[name] = dropped
        # Post-check
        s2 = dup_summary(clean, key)
        print(f"   After dedup: rows={s2['rows']:,} | unique_ids={s2['unique_ids']:,} | "
              f"duplicate_rows={s2['dup_rows']:,} | duplicate_ids={s2['dup_ids']:,}")
    else:
        print("-> No duplicates found; no action taken.")

# (Optional) reassign cleaned dataframes back to your variables
sdo1_characteristics = datasets["sdo1_characteristics"]
sdo2_skc             = datasets["sdo2_skc"]
sdo5_degree          = datasets["sdo5_degree"]
sdo6_course_results  = datasets["sdo6_course_results"]

# (Optional) inspect what was removed for any dataset:
# removed_dups["sdo6_course_results"].head()



-------------------------------- Inspect what was loaded ----------------------------------------------------------------------------

In [ ]:
# Top 50 most-null columns across all datasets
column_profile.sort_values("null_pct", ascending=False).head(50)


In [ ]:
# sdo1_characteristics dataset columns
column_profile[column_profile["dataset"] == "sdo1_characteristics"]

In [ ]:
# sdo2_skc dataset columns
column_profile[column_profile["dataset"] == "sdo2_skc"]

In [ ]:
# sdo5_degree dataset columns
column_profile[column_profile["dataset"] == "sdo5_degree"]

In [ ]:
# sdo6_course_results dataset columns
column_profile[column_profile["dataset"] == "sdo6_course_results"]

------------------------ Standardize columns before combining datasets------------------------------------------------------------

In [ ]:
# - Standardized primary key names by renaming 'sinh_d'/'sinh_id' to 'SINH_ID'.
# - Normalized column headers: collapsed whitespace into underscores
#   (e.g., "Total Credits Block A" → "Total_Credits_Block_A").
# - Preserved provenance by prefixing all non-key columns with the dataset prefix
#   (sdo1_, sdo2_, sdo5_, sdo6_); 'SINH_ID' is not prefixed.
# - Merged sdo1_characteristics, sdo2_skc, sdo5_degree, sdo6_course_results on SINH_ID
#   via left joins (sdo1 as base) to produce final DataFrame `data`.
# - Note: if any input has duplicate SINH_IDs, deduplicate/aggregate first to avoid row multiplication.



def normalize_colname(name: str) -> str:
    """
    Keep original casing, but:
      - trim
      - collapse any whitespace runs to underscores
    e.g. "Total Credits Block A" -> "Total_Credits_Block_A"
    """
    s = str(name).strip()
    s = re.sub(r"\s+", "_", s)
    return s

def unify_key_to_upper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure the primary key column is exactly 'SINH_ID'.
    Handles variants like 'sinh_d' or 'sinh_id' in any casing.
    """
    lower_map = {c.lower(): c for c in df.columns}
    if "sinh_id" in lower_map:
        return df.rename(columns={lower_map["sinh_id"]: "SINH_ID"})
    if "sinh_d" in lower_map:   # some files may have this variant
        return df.rename(columns={lower_map["sinh_d"]: "SINH_ID"})
    return df  # already has SINH_ID

def prefix_nonkey_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Add '<prefix>_' to every column EXCEPT the key 'SINH_ID'.
    """
    ren = {c: f"{prefix}_{c}" for c in df.columns if c != "SINH_ID"}
    return df.rename(columns=ren)

def prepare_df(df: pd.DataFrame, df_name: str) -> pd.DataFrame:
    """
    - Unify key name to 'SINH_ID'
    - Normalize column names (spaces -> underscores)
    - Prefix non-key columns with the first token of df_name (e.g., 'sdo1')
    """
    out = unify_key_to_upper(df)
    out = out.rename(columns=lambda c: normalize_colname(c))
    prefix = df_name.split("_", 1)[0]  # first word of the df name
    out = prefix_nonkey_columns(out, prefix)
    return out

# --- clean & prefix each table --------------------------------------------

sdo1_clean = prepare_df(sdo1_characteristics, "sdo1_characteristics")
sdo2_clean = prepare_df(sdo2_skc,             "sdo2_skc")
sdo5_clean = prepare_df(sdo5_degree,          "sdo5_degree")
sdo6_clean = prepare_df(sdo6_course_results,  "sdo6_course_results")

# (Optional) sanity checks
# print(sdo1_clean.columns.tolist())
# print(sdo2_clean.columns.tolist())
# ...

# --- combine on SINH_ID ----------------------------------------------------
# Choose sdo1 as the base
data = (
    sdo1_clean
    .merge(sdo2_clean, on="SINH_ID", how="left")
    .merge(sdo5_clean, on="SINH_ID", how="left")
    .merge(sdo6_clean, on="SINH_ID", how="left")
)

# 'data' is your final merged DataFrame
print("Final shape:", data.shape)


In [ ]:
data

----------------- Save combined data frame before further cleaning -----------------------

In [ ]:
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "v2_combined.csv"
data.to_csv(out_path, index=False)

print("Wrote:", out_path)

In [ ]:
data.columns

In [ ]:
# Drop unnecessary columns from the combined DataFrame data
data = data.drop(
    columns=[
        "sdo1_d_student_id",
        "sdo5_degree_code_letters",
        "sdo5_degree_code"
    ],
    errors="ignore"
)


In [ ]:
# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")

In [ ]:
# Check for columns with only one unique value
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]

print("Columns with only one unique value:", single_value_cols)


In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Print value counts for all columns
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
data.columns

In [ ]:
# Convert date-related columns to datetime
date_cols = [
    "sdo1_date_of_birth",
    "sdo2_SKC_DATUM",
    "sdo5_enrollment_date",
    "sdo5_degree_start_date"
]

for col in date_cols:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors="coerce")  # invalid dates -> NaT


In [ ]:
# Print value counts only for object (categorical) columns, excluding skip_cols and datetime columns
# Columns to skip manually
skip_cols = ["SINH_ID", "sdo5_POSTCODE"]

for col in data.select_dtypes(include=["object"]).columns:
    if col in skip_cols or col in data.select_dtypes(include=["datetime"]).columns:
        continue
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")


Questions: 

What is the difference between sdo1_nationality and sdo5_LAND columns
What does 0 in sdo1_gender column means

// Fraukje: LAND is the country that is registered as the student's postal address at the start of the study. I suppose it is not useful by itself, but maybe as an indicator of someone living abroad: has_foreign_address = 1 if LAND != The Netherlands else 0

In [ ]:
# Delete sdo5_LAND for now
del data['sdo5_LAND']

---------------------------------Handle categories in sdo1_nationality column ------------------------------------------------------
//I thought we agreed that we would not use Nationality as a category, but only as a bit indicating Dutch national yes/no?

In [ ]:
# Clean categorical column (sdo1_nationality):
# - Replace unknown codes ("XXX", "XX") with "Unknown"
# - Group all categories with <1% frequency into "Other"
# - Keep only dominant categories (e.g., "NL") intact


col = "sdo1_nationality"
threshold = 0.01  # 1% cutoff

# Step 1: explicitly map special unknown codes
data[col] = data[col].replace(["XXX", "XX"], "Unknown")

# Step 2: group rare categories into "Other"
freq = data[col].value_counts(normalize=True)
rare_cats = freq[freq < threshold].index
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data["sdo1_nationality"].value_counts())             # raw counts
print(data["sdo1_nationality"].value_counts(normalize=True))  # proportions


-------------------------------------- Handle categories in sdo2_SKC_RESULT Column -------------------------------------------

In [ ]:
# Clean categorical column (sdo2_SKC_RESULT):
# - Replace categories with fewer than 100 occurrences with "Other"
#// Can we keep even the small categories intact (there are only a small number of categories anyway)?

col = "sdo2_SKC_RESULT"
threshold = 100

# Find categories with counts below the threshold
value_counts = data[col].value_counts()
rare_cats = value_counts[value_counts < threshold].index

# Replace them with "Other"
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data[col].value_counts(dropna=False))

-------------------------------------- Handle categories in sdo2_ADVIES_DEF Column -------------------------------------------

In [ ]:
# Clean categorical values in sdo2_ADVIES_DEF:
# - Remove the word "met"
# - Replace spaces between remaining words with underscores
# - Preserve NaN values and original capitalization
#//Is it necessary to do cleaning? It makes the categories less interpretable for end users.
#//Can we keep even the small categories intact (there are only a small number of categories anyway)?

col = "sdo2_ADVIES_DEF"

# Apply cleaning only on non-null values
data[col] = data[col].dropna().apply(
    lambda x: "_".join([w for w in x.split() if w.lower() != "met"])
)

# Quick check
print(data[col].value_counts())


-------------------------------------- Handle categories in sdo5_degree Column -------------------------------------------

In [ ]:
# Clean categories in sdo5_degree:
# - Keep these exact mappings:
#     "B Education in Primary Schools (age 4 - 12) (day)"     -> "Primary_Education_Day"
#     "B Education in Primary Schools (age 4 - 12) (ALPO)"    -> "Primary_Education_ALPO"
#     "B Education in Primary Schools (age 4 - 12) (Regular)" -> "Primary_Education_Regular"
#     "B Arts Therapies (Music Therapy)"                      -> "Music_Therapy"
#     "B Arts Therapies (Drama Therapy)"                      -> "Drama_Therapy"
#     "B Arts Therapies (Art Therapy)"                        -> "Art_Therapy"
# - Remove leading degree prefixes: "B " and "Bachelor of "
# - Correct known typos (e.g., "Chemics" -> "Chemistry")
# - Remove prepositions between program names: "and", "with", "in" (case-insensitive)
# - Remove commas and ampersands
# - Normalize whitespace and apply Title Case
# - Convert spaces between words into underscores at the end
# - Preserve "Teacher" as-is
# - Map NaN to "Unknown"

col = "sdo5_degree"

def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()

    # ---- Exact mappings to keep as specified ----
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
    }
    if v in exact_map:
        return exact_map[v]

    # ---- Remove prefixes ----
    v = re.sub(r"^\s*B\s+", "", v)
    v = re.sub(r"^\s*Bachelor of\s+", "", v, flags=re.IGNORECASE)

    # ---- Fix known inconsistencies ----
    v = v.replace("Chemics", "Chemistry")

    # ---- Remove prepositions and punctuation ----
    v = v.replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")

    # ---- Normalize whitespace and casing ----
    v = re.sub(r"\s+", " ", v).strip()
    v = v.title()

    # ---- Replace spaces with underscores ----
    v = v.replace(" ", "_")

    return v

# Apply cleaning directly to the original column
data[col] = data[col].apply(clean_degree)

# Optional check
print(data[col].value_counts())


In [ ]:
# Save version 2 data here to later combine it with version 1
data.to_csv(os.path.join("..", "data", "processed", "dropout_version2_data.csv"), header=True)